# Problem formulation

## Setup

In [12]:
import os, sys
# ── Add JUSTICE-main to sys.path so justice internal imports resolve ───────────
_here         = os.path.abspath('../assignments_ema')   # CWD = notebook dir in JupyterLab
_justice_root = os.path.normpath(os.path.join(_here, '../JUSTICE-main'))
_NOTEBOOK_DIR = _here

_PLOTS_DIR = os.path.join(_NOTEBOOK_DIR, "../assignments_ema/plots")
os.makedirs(_PLOTS_DIR, exist_ok=True)

if _justice_root not in sys.path:
    sys.path.insert(0, _justice_root)

os.chdir(_justice_root)

import warnings
warnings.filterwarnings("ignore")

import importlib
import multiprocessing

try:
    multiprocessing.set_start_method("spawn")
except RuntimeError:
    pass  # already set

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from IPython.display import display

from ema_workbench import (
    Model, RealParameter, IntegerParameter, CategoricalParameter,
    ScalarOutcome, ArrayOutcome,
    perform_experiments, ema_logging,
    SequentialEvaluator, MultiprocessingEvaluator,
)

from justice.model import JUSTICE
from justice.util.enumerations import WelfareFunction
from justice.util.data_loader import DataLoader
from justice.objectives.objective_functions import years_above_temperature_threshold

import matplotlib.path as _mpath
def _patched_path_deepcopy(self, memo=None):
    if memo is None:
        memo = {}
    new_path = _mpath.Path.__new__(_mpath.Path)
    memo[id(self)] = new_path
    verts = self._vertices.copy()
    codes = self._codes.copy() if self._codes is not None else None
    new_path.__init__(
        verts, codes,
        _interpolation_steps=self._interpolation_steps,
        readonly=False
    )
    return new_path

_mpath.Path.__deepcopy__ = _patched_path_deepcopy

ema_logging.log_to_stderr(ema_logging.INFO)

# --- Basic model constants ---
N_REGIONS   = 57
N_TIMESTEPS = 286
N_RBFS      = 4
N_INPUTS    = 2

years   = np.arange(2015, 2301)
tau_hat = np.clip((years - 2015) / (2100 - 2015), 0, 1)
s_curve = 3 * tau_hat**2 - 2 * tau_hat**3

# --- Data loader ---
dl = DataLoader()
region_list = list(dl.REGION_LIST)
zaf_idx = region_list.index("zaf")

# --- Optional package check ---
required = [
    "justice", "numpy", "pandas", "matplotlib",
    "ema_workbench", "scipy", "seaborn"
]
for pkg in required:
    found = importlib.util.find_spec(pkg) is not None
    print(f"  {'✓' if found else '✗'}  {pkg}")

print(f"\nPython {sys.version.split()[0]}")
print(f"DataLoader ready — {len(dl.REGION_LIST)} regions, {N_TIMESTEPS} timesteps")
print(f"South Africa index: {zaf_idx}")

  ✓  justice
  ✓  numpy
  ✓  pandas
  ✓  matplotlib
  ✓  ema_workbench
  ✓  scipy
  ✓  seaborn

Python 3.12.13
DataLoader ready — 57 regions, 286 timesteps
South Africa index: 56


In [13]:
xlrm = {
    "X — Uncertainties": [
        "Future socioeconomic development pathways",
        "Climate sensitivity and climate response",
        "Regional climate damages affecting South Africa",
        "Future abatement costs",
        "Future backstop costs / clean technology costs",
        "Availability of international climate finance and Just Transition support",
    ],
    "L — Levers": [
        "Direct model levers: emission_control_rate and savings_rate",
        "Analytical lever: choice of social welfare function",
        "Political demands requiring proxies: differentiated obligations, climate finance, Just Transition support, emissions trading design",
    ],
    "R — Relationships (the model)": [
        "Climate module: emissions pathway -> temperature trajectory",
        "Damage module: temperature change -> regional economic damage",
        "Abatement module: emission control rate -> regional abatement cost",
        "Economic module: savings and output dynamics -> consumption and gross economic output",
        "Welfare aggregation: regional outcomes -> global welfare under a chosen welfare function",
    ],
    "M — Measures of performance": [
        "welfare: aggregate social welfare",
        "economic_damage: total economic damages per region",
        "damage_fraction: share of GDP lost to climate damages",
        "abatement_cost: total mitigation cost per region",
        "gross_economic_output: output before damages and abatement",
        "consumption_per_capita: net consumption per person",
        "abatement_cost / gross_economic_output for zaf: relative mitigation burden",
    ],
}

df_xlrm = pd.DataFrame(
    [(cat, item) for cat, items in xlrm.items() for item in items],
    columns=["Component", "Item"]
)

with pd.option_context("display.max_colwidth", 120):
    display(df_xlrm)

,Component,Item
0,X — Uncertainties,Future socioeconomic development pathways
1,X — Uncertainties,Climate sensitivity and climate response
2,X — Uncertainties,Regional climate damages affecting South Africa
3,X — Uncertainties,Future abatement costs
4,X — Uncertainties,Future backstop costs / clean technology costs
5,X — Uncertainties,Availability of international climate finance and Just Transition support
6,L — Levers,Direct model levers: emission_control_rate and savings_rate
7,L — Levers,Analytical lever: choice of social welfare function
8,L — Levers,"Political demands requiring proxies: differentiated obligations, climate finance, Just Transition support, emissions..."
9,R — Relationships (the model),Climate module: emissions pathway -> temperature trajectory


In [14]:
ema_mapping = pd.DataFrame([
    ("X — Uncertainties", "Socioeconomic development pathway", "CategoricalParameter (scenario)", "em_model.uncertainties"),
    ("X — Uncertainties", "Climate response uncertainty", "Climate ensemble sampling", "analysis / re-evaluation"),
    ("X — Uncertainties", "Climate damages affecting South Africa", "Captured through model outcomes", "interpreted from outputs"),
    ("X — Uncertainties", "Future abatement costs", "Captured through model structure and outcomes", "interpreted from outputs"),
    ("X — Uncertainties", "Future backstop / clean technology costs", "Conceptual uncertainty", "problem framing / proxy"),
    ("X — Uncertainties", "Availability of climate finance and Just Transition support", "Conceptual uncertainty", "problem framing / proxy"),

    ("L — Levers", "emission_control_rate", "Model input / policy lever", "em_model.levers"),
    ("L — Levers", "savings_rate", "Model input / policy lever", "em_model.levers"),
    ("L — Levers", "choice of social welfare function", "Categorical analytical lever", "model setting"),
    ("L — Levers", "Differentiated obligations / climate finance / Just Transition fund / emissions trading", "Political demands requiring proxies", "problem framing / interpretation"),

    ("M — Measures", "welfare", "ScalarOutcome", "em_model.outcomes"),
    ("M — Measures", "economic_damage", "Array or derived outcome", "model.data / post-processing"),
    ("M — Measures", "damage_fraction", "Array or derived outcome", "model.data / post-processing"),
    ("M — Measures", "abatement_cost", "Array or derived outcome", "model.data / post-processing"),
    ("M — Measures", "gross_economic_output", "Array or derived outcome", "model.data / post-processing"),
    ("M — Measures", "consumption_per_capita", "Array or derived outcome", "model.data / post-processing"),
    ("M — Measures", "abatement_cost / gross_economic_output for zaf", "Derived indicator", "post-processing"),
], columns=["XLRM", "Variable", "EMA type", "Assignment to"])

print("\nXLRM → EMA Workbench mapping:")
display(ema_mapping)


XLRM → EMA Workbench mapping:


,XLRM,Variable,EMA type,Assignment to
0,X — Uncertainties,Socioeconomic development pathway,CategoricalParameter (scenario),em_model.uncertainties
1,X — Uncertainties,Climate response uncertainty,Climate ensemble sampling,analysis / re-evaluation
2,X — Uncertainties,Climate damages affecting South Africa,Captured through model outcomes,interpreted from outputs
3,X — Uncertainties,Future abatement costs,Captured through model structure and outcomes,interpreted from outputs
4,X — Uncertainties,Future backstop / clean technology costs,Conceptual uncertainty,problem framing / proxy
5,X — Uncertainties,Availability of climate finance and Just Trans...,Conceptual uncertainty,problem framing / proxy
6,L — Levers,emission_control_rate,Model input / policy lever,em_model.levers
7,L — Levers,savings_rate,Model input / policy lever,em_model.levers
8,L — Levers,choice of social welfare function,Categorical analytical lever,model setting
9,L — Levers,Differentiated obligations / climate finance /...,Political demands requiring proxies,problem framing / interpretation


In [21]:
welfare_functions = {
    "Utilitarian"    : WelfareFunction.UTILITARIAN,
    "Prioritarian"   : WelfareFunction.PRIORITARIAN,
    "Sufficientarian": WelfareFunction.SUFFICIENTARIAN,
}

ecr_bau = np.zeros((N_REGIONS, N_TIMESTEPS))

results_wf = {}
for wf_name, wf in welfare_functions.items():
    JUSTICE.hard_reset()
    model = JUSTICE(
        start_year=2015, end_year=2300, timestep=1,
        scenario=2, climate_ensembles=1, stochastic_run=False,
        social_welfare_function=wf,
    )
    model.run(emission_control_rate=ecr_bau, endogenous_savings_rate=True)
    datasets = model.evaluate()

    wf_val = float(np.abs(datasets["welfare"]))
    yat    = float(years_above_temperature_threshold(datasets["global_temperature"], 2.0))
    _, _, _, wl_dam = model.welfare_function.calculate_welfare(
        datasets["damage_cost_per_capita"], welfare_loss=True)
    _, _, _, wl_abt = model.welfare_function.calculate_welfare(
        datasets["abatement_cost_per_capita"], welfare_loss=True)

    results_wf[wf_name] = {
        "welfare":               wf_val,
        "years_above_2C":        yat,
        "welfare_loss_damage":   float(np.abs(wl_dam)),
        "welfare_loss_abatement":float(np.abs(wl_abt)),
    }
    print(f"  {wf_name}: welfare={wf_val:.6f}, years>2°C={yat:.1f}")

df_wf = pd.DataFrame(results_wf).T.round(4)
print("\nComparison table:")
display(df_wf)

  Utilitarian: welfare=103.721111, years>2°C=259.0
  Prioritarian: welfare=414.827668, years>2°C=259.0
  Sufficientarian: welfare=104.187375, years>2°C=259.0

Comparison table:


,welfare,years_above_2C,welfare_loss_damage,welfare_loss_abatement
Utilitarian,103.7211,259.0,3980.5410,74364.1321
Prioritarian,414.8277,259.0,18888.6964,318818.3307
Sufficientarian,104.1874,259.0,3980.5410,74364.1321


In [22]:
# Run JUSTICE under BAU (no abatement) for South Africa under Prioritarian welfare

JUSTICE.hard_reset()

model = JUSTICE(
    start_year=2015,
    end_year=2300,
    timestep=1,
    scenario=2,
    climate_ensembles=1,
    stochastic_run=False,
    social_welfare_function=WelfareFunction.PRIORITARIAN,
)

# Correct dimensions from the model
n_regions = len(model.data_loader.REGION_LIST)
n_timesteps = len(model.time_horizon.model_time_horizon)
n_ensembles = model.no_of_ensembles

# BAU = zero abatement everywhere
ecr_bau = np.zeros((n_regions, n_timesteps, n_ensembles))

# Run model
model.run(ecr_bau, endogenous_savings_rate=True)
data = model.evaluate()

# South Africa index
zaf_idx = list(model.data_loader.REGION_LIST).index("zaf")

# South Africa metrics
abatement_burden_zaf = (
    data["abatement_cost"][zaf_idx, :, :] /
    data["gross_economic_output"][zaf_idx, :, :]
)

damage_burden_zaf = (
    data["economic_damage"][zaf_idx, :, :] /
    data["gross_economic_output"][zaf_idx, :, :]
)

bau_results_sa = {
    "welfare": float(np.squeeze(data["welfare"])),
    "mean_abatement_cost_zaf": float(np.nanmean(data["abatement_cost"][zaf_idx, :, :])),
    "mean_economic_damage_zaf": float(np.nanmean(data["economic_damage"][zaf_idx, :, :])),
    "mean_damage_fraction_zaf": float(np.nanmean(data["damage_fraction"][zaf_idx, :, :])),
    "mean_consumption_per_capita_zaf": float(np.nanmean(data["consumption_per_capita"][zaf_idx, :, :])),
    "mean_abatement_burden_zaf": float(np.nanmean(abatement_burden_zaf)),
    "mean_damage_burden_zaf": float(np.nanmean(damage_burden_zaf)),
    "mean_global_temperature": float(np.nanmean(data["global_temperature"])),
}

df_bau_sa = pd.DataFrame([bau_results_sa], index=["South Africa - Prioritarian BAU"])
print(df_bau_sa.round(4).to_string())

                                  welfare  mean_abatement_cost_zaf  mean_economic_damage_zaf  mean_damage_fraction_zaf  mean_consumption_per_capita_zaf  mean_abatement_burden_zaf  mean_damage_burden_zaf  mean_global_temperature
South Africa - Prioritarian BAU -414.8277                      0.0                    0.4953                    0.0773                          58.3801                        0.0                  0.0773                   4.5022


In [24]:
# --- South Africa key outputs and proxy metrics ---

# South Africa index
zaf_idx = list(model.data_loader.REGION_LIST).index("zaf")

# Core South Africa outputs
zaf_gross_output = data["gross_economic_output"][zaf_idx, :, :]
zaf_net_output = data["net_economic_output"][zaf_idx, :, :]
zaf_economic_damage = data["economic_damage"][zaf_idx, :, :]
zaf_damage_fraction = data["damage_fraction"][zaf_idx, :, :]
zaf_abatement_cost = data["abatement_cost"][zaf_idx, :, :]
zaf_consumption_pc = data["consumption_per_capita"][zaf_idx, :, :]
zaf_damage_cost_pc = data["damage_cost_per_capita"][zaf_idx, :, :]
zaf_abatement_cost_pc = data["abatement_cost_per_capita"][zaf_idx, :, :]
zaf_emissions = data["emissions"][zaf_idx, :, :]
zaf_regional_temp = data["regional_temperature"][zaf_idx, :, :]

# Key proxy metrics
zaf_abatement_burden = zaf_abatement_cost / zaf_gross_output
zaf_damage_burden = zaf_economic_damage / zaf_gross_output

# Summary table
zaf_summary = {
    "welfare": float(np.squeeze(data["welfare"])),
    "mean_gross_output_zaf": float(np.nanmean(zaf_gross_output)),
    "mean_net_output_zaf": float(np.nanmean(zaf_net_output)),
    "mean_economic_damage_zaf": float(np.nanmean(zaf_economic_damage)),
    "mean_damage_fraction_zaf": float(np.nanmean(zaf_damage_fraction)),
    "mean_abatement_cost_zaf": float(np.nanmean(zaf_abatement_cost)),
    "mean_consumption_per_capita_zaf": float(np.nanmean(zaf_consumption_pc)),
    "mean_damage_cost_per_capita_zaf": float(np.nanmean(zaf_damage_cost_pc)),
    "mean_abatement_cost_per_capita_zaf": float(np.nanmean(zaf_abatement_cost_pc)),
    "mean_emissions_zaf": float(np.nanmean(zaf_emissions)),
    "mean_regional_temperature_zaf": float(np.nanmean(zaf_regional_temp)),
    "mean_global_temperature": float(np.nanmean(data["global_temperature"])),
    "mean_abatement_burden_zaf": float(np.nanmean(zaf_abatement_burden)),
    "max_abatement_burden_zaf": float(np.nanmax(zaf_abatement_burden)),
    "mean_damage_burden_zaf": float(np.nanmean(zaf_damage_burden)),
    "max_damage_burden_zaf": float(np.nanmax(zaf_damage_burden)),
}

df_zaf_summary = pd.DataFrame([zaf_summary], index=["South Africa"])
display(df_zaf_summary.round(4))

,welfare,mean_gross_output_zaf,mean_net_output_zaf,mean_economic_damage_zaf,mean_damage_fraction_zaf,mean_abatement_cost_zaf,mean_consumption_per_capita_zaf,mean_damage_cost_per_capita_zaf,mean_abatement_cost_per_capita_zaf,mean_emissions_zaf,mean_regional_temperature_zaf,mean_global_temperature,mean_abatement_burden_zaf,max_abatement_burden_zaf,mean_damage_burden_zaf,max_damage_burden_zaf
South Africa,-414.8277,5.2839,4.7886,0.4953,0.0773,0.0,58.3801,8.1198,0.0,0.2212,21.183,4.5022,0.0,0.0,0.0773,0.1155


In [27]:
# --- Compare South Africa with more relevant regions ---

compare_regions = [
    # South Africa + Africa / regional peers
    "zaf", "rsaf", "egy", "noan", "noap",

    # More comparable emerging / transition economies
    "pol", "tur", "mex", "bra", "idn", "chn", "rus",

    # Developed reference group
    "usa", "gbr", "fra", "rfa"
]

rows = []
for region in compare_regions:
    idx = list(model.data_loader.REGION_LIST).index(region)

    gross_output = data["gross_economic_output"][idx, :, :]
    net_output = data["net_economic_output"][idx, :, :]
    economic_damage = data["economic_damage"][idx, :, :]
    damage_fraction = data["damage_fraction"][idx, :, :]
    abatement_cost = data["abatement_cost"][idx, :, :]
    consumption_pc = data["consumption_per_capita"][idx, :, :]

    abatement_burden = abatement_cost / gross_output
    damage_burden = economic_damage / gross_output

    rows.append({
        "region": region,
        "mean_gross_output": float(np.nanmean(gross_output)),
        "mean_net_output": float(np.nanmean(net_output)),
        "mean_abatement_cost": float(np.nanmean(abatement_cost)),
        "mean_economic_damage": float(np.nanmean(economic_damage)),
        "mean_damage_fraction": float(np.nanmean(damage_fraction)),
        "mean_consumption_per_capita": float(np.nanmean(consumption_pc)),
        "mean_abatement_burden": float(np.nanmean(abatement_burden)),
        "max_abatement_burden": float(np.nanmax(abatement_burden)),
        "mean_damage_burden": float(np.nanmean(damage_burden)),
        "max_damage_burden": float(np.nanmax(damage_burden)),
    })

df_compare = pd.DataFrame(rows)

# Optional: label comparison groups
group_map = {
    "zaf": "South Africa",
    "rsaf": "Africa / regional peer",
    "egy": "Africa / regional peer",
    "noan": "Africa / regional peer",
    "noap": "Africa / regional peer",
    "pol": "Coal / emerging comparator",
    "tur": "Coal / emerging comparator",
    "mex": "Coal / emerging comparator",
    "bra": "Coal / emerging comparator",
    "idn": "Coal / emerging comparator",
    "chn": "Coal / emerging comparator",
    "rus": "Coal / emerging comparator",
    "usa": "Developed reference",
    "gbr": "Developed reference",
    "fra": "Developed reference",
    "rfa": "Developed reference",
}

df_compare["group"] = df_compare["region"].map(group_map)

# Sort by relative abatement burden
df_compare = df_compare.sort_values("mean_abatement_burden", ascending=False).reset_index(drop=True)

display(df_compare.round(4))

,region,mean_gross_output,mean_net_output,mean_abatement_cost,mean_economic_damage,mean_damage_fraction,mean_consumption_per_capita,mean_abatement_burden,max_abatement_burden,mean_damage_burden,max_damage_burden,group
0,zaf,5.2839,4.7886,0.0,0.4953,0.0773,58.3801,0.0,0.0,0.0773,0.1155,South Africa
1,rsaf,268.2783,230.0742,0.0,38.2041,0.1091,63.9895,0.0,0.0,0.1091,0.1608,Africa / regional peer
2,egy,13.2796,11.6657,0.0,1.6139,0.0994,62.4722,0.0,0.0,0.0994,0.1471,Africa / regional peer
3,noan,4.0105,3.5798,0.0,0.4307,0.0888,74.8134,0.0,0.0,0.0888,0.1325,Africa / regional peer
4,noap,5.0872,4.5220,0.0,0.5653,0.0886,59.6208,0.0,0.0,0.0886,0.1322,Africa / regional peer
5,pol,2.1513,2.0628,0.0,0.0885,0.0369,67.8416,0.0,0.0,0.0369,0.0583,Coal / emerging comparator
6,tur,6.4781,6.0325,0.0,0.4456,0.0606,57.8787,0.0,0.0,0.0606,0.0922,Coal / emerging comparator
7,mex,11.4855,10.3334,0.0,1.1521,0.0849,69.6476,0.0,0.0,0.0849,0.1262,Coal / emerging comparator
8,bra,11.0141,9.7454,0.0,1.2687,0.0982,50.6745,0.0,0.0,0.0982,0.1449,Coal / emerging comparator
9,idn,20.5500,18.2030,0.0,2.3470,0.0953,59.3762,0.0,0.0,0.0953,0.1400,Coal / emerging comparator


In [26]:
# --- South Africa satisficing criterion ---

def south_africa_satisficing(
    row_or_metrics,
    max_mean_abatement_burden=0.05,
    min_mean_consumption_per_capita=5.0,
):
    """
    Simple satisficing rule for South Africa.
    Returns True if the policy is acceptable under the chosen thresholds.
    """
    return (
        row_or_metrics["mean_abatement_burden"] <= max_mean_abatement_burden
        and row_or_metrics["mean_consumption_per_capita"] >= min_mean_consumption_per_capita
    )